# Residência em IA — Rodada Pós-Gate 9 (Multi-Seed & Novos Modelos)

Notebook preparado para execução em **Google Colab (A100 80GB / G6 80+ GB)** com:
- dependências exatas e preflight de GPU, token e acesso aos modelos;
- retomada automática via `resume_journal.py` salva diretamente no **Google Drive**;
- smoke obrigatório em ordem crescente de custo;
- Blocos: `smoke` (24 itens), `bloco_a` (30 runs, 5 seeds no mesmo ambiente), `bloco_b` (63 runs) e `bloco_c` (9 runs do Llama 3.3 70B INT8).

Leia `colab/README.md` antes da execução completa.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive, userdata
import os
from pathlib import Path
import subprocess
import sys

drive.mount('/content/drive')
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Configure HF_TOKEN nos Secrets do Colab antes de continuar')
os.environ['HF_TOKEN'] = hf_token
OUTPUT_ROOT = '/content/drive/MyDrive/residencia_BAP_colab/results_colab'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
# Clonar ou atualizar o repositório e registrar o commit executado
REPO_DIR = '/content/residencia_run1_BAP'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/motafernando/residencia_run1_BAP.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('REPO_COMMIT:', REPO_COMMIT)

In [ ]:
# Instalar o ambiente exato. O preflight seguinte confirma as versões efetivas.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                '-r', 'colab/requirements.txt'], check=True)
print('PIP_LOCK_OK')

In [ ]:
# Preflight obrigatório: versões, A100 80 GB e acesso real a cada config.json do smoke.
PREFLIGHT_REPORT = str(Path(OUTPUT_ROOT) / 'preflight-smoke.json')
subprocess.run([sys.executable, 'colab/preflight.py', '--block', 'smoke',
                '--report', PREFLIGHT_REPORT], check=True)

In [ ]:
# 1) Smoke test obrigatório (24 itens) para validar templates, enable_thinking=False e VRAM do 70B INT8
subprocess.run([sys.executable, 'colab/run_colab_grid.py', '--block', 'smoke',
                '--output-root', OUTPUT_ROOT], check=True)

In [ ]:
# 2) Bloco A — 5 seeds completas no mesmo ambiente Colab (2026-2030)
subprocess.run([sys.executable, 'colab/run_colab_grid.py', '--block', 'bloco_a',
                '--output-root', OUTPUT_ROOT], check=True)

In [ ]:
# 3) Bloco B — Últimos pesos e modelos equivalentes (~8B)
# Dica: você pode filtrar por modelo, ex: --filter qwen3_8b
subprocess.run([sys.executable, 'colab/run_colab_grid.py', '--block', 'bloco_b',
                '--output-root', OUTPUT_ROOT], check=True)

In [ ]:
# 4) Bloco C — Llama 3.3 70B INT8 (requer 80+ GB VRAM)
subprocess.run([sys.executable, 'colab/run_colab_grid.py', '--block', 'bloco_c',
                '--output-root', OUTPUT_ROOT], check=True)

In [ ]:
# 5) Consolidação final. Smokes são excluídos automaticamente.
CONSOLIDATED = str(Path(OUTPUT_ROOT) / 'consolidado_v15')
subprocess.run([sys.executable, 'analises/consolidar_multiseed.py',
                '--results-root', OUTPUT_ROOT, '--output-dir', CONSOLIDATED], check=True)